# Brian MF Transfer-Function Quickstart

This notebook demonstrates a minimal parity-first workflow for the new `tvbtoolkit.brian_mf` module:

1. Create synthetic transfer-function data
2. Fit AdEx transfer-function coefficients
3. Visualise fit quality
4. Save fitted parameters to the parameter database

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from tvbtoolkit.brian_mf.mean_field.tf_calc import (
    TransferFunctionFitConfig,
    fit_adex_transfer_function,
    get_neuron_params_double_cell,
    save_param_set,
)
from tvbtoolkit.brian_mf.parity.fixtures import fixed_rate_grid

FIG_DIR = Path("./results/figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
ve, vi, empirical, adapt = fixed_rate_grid()
params = get_neuron_params_double_cell("FS-RS", si_units=False)

cfg = TransferFunctionFitConfig(loop_n=3, tf_maxiter=500, vthr_maxiter=500)
fit = fit_adex_transfer_function(
    empirical,
    {
        "ve": ve,
        "vi": vi,
        "adapt": adapt,
        "params": params,
        "cell_type": "RS",
        "w_prec": False,
    },
    cfg,
)
fit.rmse_hz, fit.fitted_params


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

im0 = axes[0].imshow(empirical, aspect="auto", origin="lower")
axes[0].set_title("Empirical TF")
axes[0].set_xlabel("ve index")
axes[0].set_ylabel("vi index")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(fit.fit_rate, aspect="auto", origin="lower")
axes[1].set_title("Fitted TF")
axes[1].set_xlabel("ve index")
axes[1].set_ylabel("vi index")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

residual = empirical - fit.fit_rate
im2 = axes[2].imshow(residual, aspect="auto", origin="lower", cmap="coolwarm")
axes[2].set_title("Residuals")
axes[2].set_xlabel("ve index")
axes[2].set_ylabel("vi index")
fig.colorbar(im2, ax=axes[2], fraction=0.046)

out = FIG_DIR / "brain_mf_quickstart_fit.svg"
fig.savefig(out, dpi=300)
out


In [ ]:
metadata = {
    "cell_type": "RS",
    "species": "synthetic-demo",
    "temperature": "N/A",
    "recording_condition": "in_silico",
    "source": "TVBToolkit brian_mf quickstart",
    "toolbox_version": "0.1.0",
    "date": "2026-02-20",
}

path = save_param_set(
    name="quickstart_rs_demo",
    params={f"P{i}": float(v) for i, v in enumerate(fit.fitted_params)},
    metadata=metadata,
    path="./data/brian_mf/param_db",
)
path
